# AIC 2026 — Tải toàn bộ keyframe lên Google Drive

Mở notebook bằng **Google Colab**, chọn **Runtime → Run all**, rồi cấp quyền Google Drive.

Notebook này tải 14 ZIP keyframe chính thức L21–L30 vào:

`MyDrive/AIC2026/keyframe_zips/`

Tổng dung lượng chính xác: **30.810 GB (28.694 GiB)**. File đang tải dùng đuôi `.part`; nếu Colab bị ngắt, hãy chạy lại notebook để tiếp tục. Notebook tự bỏ qua file đã đủ dung lượng và đọc được cấu trúc ZIP.

> Chưa giải nén keyframe. Giữ nguyên ZIP trên Drive để tránh tạo hàng trăm nghìn file nhỏ và làm Drive/Colab chậm.


In [1]:
# 1) Kết nối Google Drive và tạo thư mục đích
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        'Hãy mở notebook này bằng Google Colab rồi chọn Runtime > Run all.'
    ) from exc

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AIC2026')
KEYFRAME_ZIP_DIR = DRIVE_ROOT / 'keyframe_zips'
KEYFRAME_ZIP_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root       : {DRIVE_ROOT}')
print(f'Keyframe ZIP dir : {KEYFRAME_ZIP_DIR}')


Mounted at /content/drive
Drive root       : /content/drive/MyDrive/AIC2026
Keyframe ZIP dir : /content/drive/MyDrive/AIC2026/keyframe_zips


In [2]:
# 2) Manifest chính thức — đã kiểm tra HTTP 200 ngày 28/08/2026
MANIFEST = [
    {'name': 'Keyframes_L21.zip',   'bytes': 1447092929},
    {'name': 'Keyframes_L22.zip',   'bytes': 1720707161},
    {'name': 'Keyframes_L23.zip',   'bytes':  506221125},
    {'name': 'Keyframes_L24.zip',   'bytes': 1726819558},
    {'name': 'Keyframes_L25.zip',   'bytes': 6092515522},
    {'name': 'Keyframes_L26_a.zip', 'bytes': 2298526063},
    {'name': 'Keyframes_L26_b.zip', 'bytes': 2418048525},
    {'name': 'Keyframes_L26_c.zip', 'bytes': 2479413533},
    {'name': 'Keyframes_L26_d.zip', 'bytes': 2474966491},
    {'name': 'Keyframes_L26_e.zip', 'bytes': 2468869815},
    {'name': 'Keyframes_L27.zip',   'bytes': 1091049736},
    {'name': 'Keyframes_L28.zip',   'bytes': 2164939460},
    {'name': 'Keyframes_L29.zip',   'bytes': 2509637849},
    {'name': 'Keyframes_L30.zip',   'bytes': 1411485866},
]

BASE_URL = 'https://aic-data.ledo.io.vn'
for item in MANIFEST:
    item['url'] = f"{BASE_URL}/{item['name']}"

expected_total = sum(item['bytes'] for item in MANIFEST)
assert len(MANIFEST) == 14
assert expected_total == 30_810_293_633
print(f'{len(MANIFEST)} ZIP | {expected_total / 1e9:.3f} GB | {expected_total / 2**30:.3f} GiB')


14 ZIP | 30.810 GB | 28.694 GiB


In [3]:
# 3) Xem trạng thái hiện tại trước khi tải
import zipfile
import pandas as pd

def zip_structure_ok(path):
    try:
        with zipfile.ZipFile(path) as archive:
            return len(archive.infolist()) > 0
    except (OSError, zipfile.BadZipFile):
        return False

def inspect_item(item, check_zip=True):
    final_path = KEYFRAME_ZIP_DIR / item['name']
    part_path = KEYFRAME_ZIP_DIR / f"{item['name']}.part"
    expected = item['bytes']

    if final_path.exists():
        actual = final_path.stat().st_size
        if actual == expected and (not check_zip or zip_structure_ok(final_path)):
            status = 'HOÀN TẤT'
        elif actual == expected:
            status = 'ZIP LỖI'
        else:
            status = 'SAI DUNG LƯỢNG'
    elif part_path.exists():
        actual = part_path.stat().st_size
        status = f'ĐANG DỞ {100 * actual / expected:.1f}%'
    else:
        actual = 0
        status = 'CHƯA TẢI'

    return {
        'file': item['name'],
        'expected_GiB': round(expected / 2**30, 3),
        'present_GiB': round(actual / 2**30, 3),
        'status': status,
    }

status_before = pd.DataFrame(inspect_item(item) for item in MANIFEST)
display(status_before)
done_before = sum(row == 'HOÀN TẤT' for row in status_before['status'])
print(f'Đã hoàn tất: {done_before}/{len(MANIFEST)} ZIP')

# DriveFS đôi khi báo quota chưa chính xác, nên dòng này chỉ để tham khảo.
import shutil
drive_usage = shutil.disk_usage('/content/drive/MyDrive')
remaining_bytes = sum(
    item['bytes'] for item in MANIFEST
    if inspect_item(item, check_zip=False)['status'] != 'HOÀN TẤT'
)
print(f'DriveFS báo còn trống khoảng: {drive_usage.free / 2**30:.1f} GiB')
print(f'Dữ liệu keyframe tối đa còn cần: {remaining_bytes / 2**30:.1f} GiB')
if drive_usage.free < remaining_bytes:
    print('⚠️ Số liệu DriveFS báo thiếu chỗ. Nếu gói 2 TB đã kích hoạt, hãy vẫn kiểm tra quota trên drive.google.com trước khi tải.')


,file,expected_GiB,present_GiB,status
0,Keyframes_L21.zip,1.348,0.0,CHƯA TẢI
1,Keyframes_L22.zip,1.603,0.0,CHƯA TẢI
2,Keyframes_L23.zip,0.471,0.0,CHƯA TẢI
3,Keyframes_L24.zip,1.608,0.0,CHƯA TẢI
4,Keyframes_L25.zip,5.674,0.0,CHƯA TẢI
5,Keyframes_L26_a.zip,2.141,0.0,CHƯA TẢI
6,Keyframes_L26_b.zip,2.252,0.0,CHƯA TẢI
7,Keyframes_L26_c.zip,2.309,0.0,CHƯA TẢI
8,Keyframes_L26_d.zip,2.305,0.0,CHƯA TẢI
9,Keyframes_L26_e.zip,2.299,0.0,CHƯA TẢI


Đã hoàn tất: 0/14 ZIP
DriveFS báo còn trống khoảng: 28.1 GiB
Dữ liệu keyframe tối đa còn cần: 28.7 GiB
⚠️ Số liệu DriveFS báo thiếu chỗ. Nếu gói 2 TB đã kích hoạt, hãy vẫn kiểm tra quota trên drive.google.com trước khi tải.


In [4]:
# 4) Cài aria2 — trình tải nhanh, hỗ trợ tiếp tục file dở
import shutil
import subprocess

if shutil.which('aria2c') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2'], check=True)

aria2_path = shutil.which('aria2c')
if aria2_path is None:
    raise RuntimeError('Không cài được aria2c. Hãy chạy lại cell này.')
print(f'aria2c: {aria2_path}')


aria2c: /usr/bin/aria2c


In [5]:
# 5) BẮT ĐẦU TẢI — có thể chạy lại cell này để resume
from datetime import datetime
import os
import time

def preserve_invalid(path):
    if not path.exists():
        return None
    stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    backup = path.with_name(f'{path.name}.invalid-{stamp}')
    path.rename(backup)
    print(f'  ! Giữ lại file lỗi tại: {backup.name}')
    return backup

def download_one(item):
    name = item['name']
    expected = item['bytes']
    final_path = KEYFRAME_ZIP_DIR / name
    part_path = KEYFRAME_ZIP_DIR / f'{name}.part'

    if final_path.exists():
        if final_path.stat().st_size == expected and zip_structure_ok(final_path):
            print(f'✓ Bỏ qua {name}: đã hoàn tất')
            return True
        preserve_invalid(final_path)

    if part_path.exists() and part_path.stat().st_size > expected:
        preserve_invalid(part_path)

    resumed = part_path.stat().st_size if part_path.exists() else 0
    print('\n' + '=' * 72)
    print(f'↓ {name} | {expected / 2**30:.3f} GiB | tiếp tục từ {resumed / 2**30:.3f} GiB')

    command = [
        aria2_path,
        '--continue=true',
        # Một luồng ghi tuần tự an toàn nhất với Google DriveFS và resume.
        '--max-connection-per-server=1',
        '--split=1',
        '--min-split-size=16M',
        '--file-allocation=none',
        '--auto-file-renaming=false',
        '--allow-overwrite=true',
        '--max-tries=20',
        '--retry-wait=5',
        '--connect-timeout=30',
        '--timeout=60',
        '--summary-interval=30',
        '--console-log-level=notice',
        f'--dir={KEYFRAME_ZIP_DIR}',
        f'--out={part_path.name}',
        item['url'],
    ]
    result = subprocess.run(command)
    if result.returncode != 0:
        print(f'✗ aria2c dừng với mã {result.returncode}; file .part vẫn được giữ để resume.')
        return False

    if not part_path.exists():
        print('✗ Không tìm thấy file .part sau khi tải.')
        return False

    actual = part_path.stat().st_size
    if actual != expected:
        print(f'✗ Sai dung lượng: {actual:,} / {expected:,} bytes; giữ .part để tải tiếp.')
        return False
    if not zip_structure_ok(part_path):
        print('✗ Không đọc được cấu trúc ZIP; giữ file để kiểm tra, chưa đổi tên.')
        return False

    part_path.replace(final_path)
    aria_sidecar = KEYFRAME_ZIP_DIR / f'{part_path.name}.aria2'
    if aria_sidecar.exists():
        aria_sidecar.unlink()
    print(f'✓ HOÀN TẤT: {name}')
    return True

results = {}
for index, item in enumerate(MANIFEST, start=1):
    print(f'\n[{index}/{len(MANIFEST)}]')
    try:
        results[item['name']] = download_one(item)
    except KeyboardInterrupt:
        print('\nĐã dừng theo yêu cầu. Chạy lại cell 5 để tiếp tục từ file .part.')
        raise
    except Exception as exc:
        results[item['name']] = False
        print(f'✗ Lỗi {type(exc).__name__}: {exc}')
        print('  Notebook sẽ chuyển sang file kế tiếp; sau đó chạy lại cell 5 để thử lại.')
    time.sleep(1)



[1/14]

↓ Keyframes_L21.zip | 1.348 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L21.zip

[2/14]

↓ Keyframes_L22.zip | 1.603 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L22.zip

[3/14]

↓ Keyframes_L23.zip | 0.471 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L23.zip

[4/14]

↓ Keyframes_L24.zip | 1.608 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L24.zip

[5/14]

↓ Keyframes_L25.zip | 5.674 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L25.zip

[6/14]

↓ Keyframes_L26_a.zip | 2.141 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L26_a.zip

[7/14]

↓ Keyframes_L26_b.zip | 2.252 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L26_b.zip

[8/14]

↓ Keyframes_L26_c.zip | 2.309 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L26_c.zip

[9/14]

↓ Keyframes_L26_d.zip | 2.305 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L26_d.zip

[10/14]

↓ Keyframes_L26_e.zip | 2.299 GiB | tiếp tục từ 0.000 GiB
✓ HOÀN TẤT: Keyframes_L26_e.zip

[11/14]

↓ Keyframes_L27.zip

In [6]:
# 6) Báo cáo cuối — Run all sẽ tự chạy cell này sau khi tải xong
import json

status_after = pd.DataFrame(inspect_item(item) for item in MANIFEST)
display(status_after)
done_after = sum(row == 'HOÀN TẤT' for row in status_after['status'])
present_bytes = sum(
    (KEYFRAME_ZIP_DIR / item['name']).stat().st_size
    for item in MANIFEST
    if (KEYFRAME_ZIP_DIR / item['name']).exists()
)

report = {
    'checked_at': datetime.now().isoformat(timespec='seconds'),
    'target_dir': str(KEYFRAME_ZIP_DIR),
    'complete_files': int(done_after),
    'total_files': len(MANIFEST),
    'expected_total_bytes': expected_total,
    'complete_file_bytes': int(present_bytes),
    'files': status_after.to_dict(orient='records'),
}
with open(KEYFRAME_ZIP_DIR / 'download_status.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, ensure_ascii=False, indent=2)

if done_after == len(MANIFEST):
    print('\n✅ ĐÃ TẢI ĐỦ 14/14 KEYFRAME ZIP. KHÔNG CẦN GIẢI NÉN LÚC NÀY.')
else:
    missing = status_after.loc[status_after['status'] != 'HOÀN TẤT', 'file'].tolist()
    print(f'\n⚠️ Mới hoàn tất {done_after}/{len(MANIFEST)} ZIP.')
    print('Chạy lại cell 5, rồi cell 6 để tiếp tục. Còn thiếu:')
    for name in missing:
        print(f'  - {name}')


,file,expected_GiB,present_GiB,status
0,Keyframes_L21.zip,1.348,1.348,HOÀN TẤT
1,Keyframes_L22.zip,1.603,1.603,HOÀN TẤT
2,Keyframes_L23.zip,0.471,0.471,HOÀN TẤT
3,Keyframes_L24.zip,1.608,1.608,HOÀN TẤT
4,Keyframes_L25.zip,5.674,5.674,HOÀN TẤT
5,Keyframes_L26_a.zip,2.141,2.141,HOÀN TẤT
6,Keyframes_L26_b.zip,2.252,2.252,HOÀN TẤT
7,Keyframes_L26_c.zip,2.309,2.309,HOÀN TẤT
8,Keyframes_L26_d.zip,2.305,2.305,HOÀN TẤT
9,Keyframes_L26_e.zip,2.299,2.299,HOÀN TẤT



✅ ĐÃ TẢI ĐỦ 14/14 KEYFRAME ZIP. KHÔNG CẦN GIẢI NÉN LÚC NÀY.


## Sau khi tải

Khi thấy dòng `ĐÃ TẢI ĐỦ 14/14 KEYFRAME ZIP`, bạn có thể đóng Colab. Không bấm giải nén toàn bộ. Khi có truy vấn cần kiểm tra bằng mắt, ta chỉ giải nén ZIP của nhóm `Lxx` liên quan.
